In [ ]:
import os
import json
import requests

host = spark.conf.get("spark.databricks.workspaceUrl")

token = dbutils.notebook.entry_point.getDbutils() \
    .notebook().getContext().apiToken().get()

headers = {
    "Authorization": f"Bearer {token}"
}

In [ ]:
existing_jobs = requests.get(
    f"https://{host}/api/2.2/jobs/list",
    headers=headers
).json().get("jobs", [])

existing_job_names = {
    j["settings"]["name"]: j["job_id"]
    for j in existing_jobs
}

In [ ]:
job_folder = "/Workspace/Repos/filipshahanski@gmail.com/dbx-ingestion-framework/config/jobs"

files = [f.path.replace("dbfs:", "") for f in dbutils.fs.ls(job_folder) if f.path.endswith(".json")]

In [ ]:
for file_path in files:
    print(f"Processing {file_path}")

    # read file
    local_path = file_path.replace("/Workspace", "/dbfs")
    
    with open(file_path, "r") as f:
        job_def = json.load(f)

    job_name = job_def["name"]

    # check if exists
    if job_name in existing_job_names:
        print(f"Skipping (exists): {job_name}")
        continue

    # create job
    response = requests.post(
        f"https://{host}/api/2.2/jobs/create",
        headers=headers,
        json=job_def
    )

    print(f"Created: {job_name} -> {response.status_code}")